# Cell-Specific LDSC

Goal: estimate cell-specific enrichment by running partitioned LDSC with one query annotation per cell type.

This notebook uses a tiny synthetic dataset so every cell can run as a smoke test. For real analysis, build query annotations from cell-type BED files or `.annot.gz` files, compute baseline-plus-query LD scores, and run `partitioned-h2`. Query annotations require explicit baseline annotations; the synthetic all-ones `base` path is only for `h2`/`rg` runs with no query inputs. `partitioned-h2` rejects baseline-only LD-score directories.

Output directories are created when missing and reused when present. For `munge-sumstats`, `ldscore`, `partitioned-h2`, and `annotate`, existing owned workflow artifacts are refused before writing unless you pass `--overwrite` or `overwrite=True`. Successful overwrites remove stale siblings from the same workflow family, such as old query shards, `ldscore.query.parquet`, or `diagnostics/query_annotations/`. The toy CLI path writes root `metadata.json` beside the temporary `sumstats.parquet` file so the regression command reloads current sumstats provenance.

In [ ]:
from pathlib import Path
import sys

def find_src_dir(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "ldsc").exists():
            return candidate / "src"
        nested = candidate / "ldsc_py3_restructured" / "src"
        if (nested / "ldsc").exists():
            return nested
    raise FileNotFoundError("Could not find src/ldsc from the current working directory.")

SRC_DIR = find_src_dir(Path.cwd())
REPO_DIR = SRC_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(SRC_DIR)

In [ ]:
import os
import subprocess
import tempfile

import numpy as np
import pandas as pd

from ldsc import (
    GlobalConfig,
    LDScoreDirectoryWriter,
    LDScoreOutputConfig,
    LDScoreResult,
    RegressionConfig,
    RegressionRunner,
    SumstatsMunger,
    set_global_config,
)
from ldsc.annotation_builder import AnnotationBundle
from ldsc.sumstats_munger import SumstatsTable


## Python API

Cell-specific LDSC is represented as partitioned h2 over query annotation columns. The runner tests each query column in a baseline-plus-one-query model.

In [ ]:
GLOBAL_CONFIG = GlobalConfig(snp_identifier="chr_pos_allele_aware", genome_build="hg38")
set_global_config(GLOBAL_CONFIG)

n_snps = 20
snps = [f"rs{i}" for i in range(1, n_snps + 1)]
index = np.arange(n_snps)
positions = np.arange(100, 100 + n_snps)
identity_snps = frozenset(f"1:{pos}:A:G" for pos in positions)

sumstats = SumstatsTable(
    data=pd.DataFrame(
        {
            "SNP": snps,
            "CHR": ["1"] * n_snps,
            "POS": positions,
            "Z": np.linspace(-2.0, 2.0, n_snps) + 0.1 * np.sin(index),
            "N": np.full(n_snps, 10000.0),
            "A1": ["A"] * n_snps,
            "A2": ["G"] * n_snps,
        }
    ),
    has_alleles=True,
    source_path="synthetic_trait",
    trait_name="trait",
    provenance={},
    config_snapshot=GLOBAL_CONFIG,
)

baseline_table = pd.DataFrame(
    {
        "CHR": ["1"] * n_snps,
        "SNP": snps,
        "POS": positions,
        "A1": ["A"] * n_snps,
        "A2": ["G"] * n_snps,
        "regression_ld_scores": np.linspace(1.1, 1.5, n_snps),
        "base": np.linspace(1.0, 2.0, n_snps),
    }
)
query_table = pd.DataFrame(
    {
        "CHR": ["1"] * n_snps,
        "SNP": snps,
        "POS": positions,
        "A1": ["A"] * n_snps,
        "A2": ["G"] * n_snps,
        "neuron": np.linspace(0.2, 1.7, n_snps),
        "glia": np.linspace(1.5, 0.3, n_snps),
    }
)
ldscore_result = LDScoreResult(
    baseline_table=baseline_table,
    query_table=query_table,
    count_records=[
        {
            "group": "baseline",
            "column": "base",
            "all_reference_snp_count": 120.0,
            "common_reference_snp_count": 100.0,
        },
        {
            "group": "query",
            "column": "neuron",
            "all_reference_snp_count": 50.0,
            "common_reference_snp_count": 30.0,
        },
        {
            "group": "query",
            "column": "glia",
            "all_reference_snp_count": 60.0,
            "common_reference_snp_count": 40.0,
        },
    ],
    baseline_columns=["base"],
    query_columns=["neuron", "glia"],
    ld_reference_snps=identity_snps,
    ld_regression_snps=identity_snps,
    chromosome_results=[],
    config_snapshot=GLOBAL_CONFIG,
)

annotation_bundle = AnnotationBundle(
    metadata=pd.DataFrame(
        {
            "CHR": ["1"] * n_snps,
            "POS": positions,
            "SNP": snps,
            "CM": np.linspace(0.0, 0.2, n_snps),
        }
    ),
    baseline_annotations=pd.DataFrame({"base": np.ones(n_snps)}),
    query_annotations=pd.DataFrame(
        {
            "neuron": (index % 2 == 0).astype(int),
            "glia": (index % 2 == 1).astype(int),
        }
    ),
    baseline_columns=["base"],
    query_columns=["neuron", "glia"],
    chromosomes=["1"],
    source_summary={},
    config_snapshot=GLOBAL_CONFIG,
)

baseline_table.head()

In [ ]:
runner = RegressionRunner(
    global_config=GLOBAL_CONFIG,
    regression_config=RegressionConfig(n_blocks=10, use_intercept=False),
)
cell_specific = runner.estimate_partitioned_h2_batch(
    sumstats,
    ldscore_result,
    annotation_bundle,
)
cell_specific

## CLI

For real inputs, compute LD scores with `ldsc ldscore --baseline-annot-sources ... --query-annot-bed-sources ... --output-dir ...`, then run `ldsc partitioned-h2 --ldscore-dir ... --output-dir ...`. Do not omit `--baseline-annot-sources` when query annotations are supplied. Add `--overwrite` only for intentional reruns; a successful overwrite removes stale owned siblings not produced by that run, including stale `ldscore.query.parquet` or `diagnostics/query_annotations/` when switching to a narrower output mode. The cell below writes the toy tables to a canonical LD-score result directory and runs the same regression CLI path. The LD-score parquet files remain flat, with chromosome-aligned row groups recorded in root `metadata.json`.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir_raw:
    tmpdir = Path(tmpdir_raw)
    sumstats_dir = tmpdir / "trait"
    ldscore_dir = tmpdir / "cell_specific_ldscores"
    output_dir = tmpdir / "cell_specific"

    sumstats_file = Path(SumstatsMunger().write_output(sumstats, sumstats_dir, overwrite=True))
    LDScoreDirectoryWriter().write(
        ldscore_result,
        LDScoreOutputConfig(output_dir=ldscore_dir, overwrite=True),
    )

    env = os.environ.copy()
    env["PYTHONPATH"] = str(SRC_DIR)
    command = [
        sys.executable,
        "-m",
        "ldsc",
        "partitioned-h2",
        "--sumstats-file",
        str(sumstats_file),
        "--trait-name",
        "trait",
        "--ldscore-dir",
        str(ldscore_dir),
        "--count-kind",
        "common",
        "--n-blocks",
        "10",
        "--no-intercept",
        "--output-dir",
        str(output_dir),
        "--write-per-query-results",
    ]
    completed = subprocess.run(command, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(f"ldsc partitioned-h2 failed with exit code {completed.returncode}")
    cli_summary = pd.read_csv(output_dir / "partitioned_h2.tsv", sep="\t")
    per_query_manifest = pd.read_csv(output_dir / "diagnostics" / "query_annotations" / "manifest.tsv", sep="\t")

cli_summary, per_query_manifest
